In [2]:
import sys
import os

# Add the src/ directory to the Python path
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "../src/")))

In [3]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import pandas_udf, PandasUDFType
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, ArrayType, MapType
from schema_definition import get_subject_schema, get_feature_schema
from feature_extraction import processEpoch, processSub
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, IntegerType
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql import DataFrame

In [4]:
TYPE = '_test_m1'

alz_df_test_m1 = pd.read_pickle(f"eeg_features_alzheimers{TYPE}.pkl")
cntrl_df_test_m1 = pd.read_pickle(f"eeg_features_controls{TYPE}.pkl")

In [5]:
alz_df_test_m1.head()

,SubjectID,EpochID,WaveBand,Electrode,Power
0,sub-020,ep-0,Delta,Fp1,0.078234
1,sub-020,ep-0,Theta,Fp1,0.006522
2,sub-020,ep-0,Alpha,Fp1,0.003593
3,sub-020,ep-0,Beta,Fp1,0.000334
4,sub-020,ep-0,Total,Fp1,0.011236


In [6]:
alz_df_test_laptop = pd.read_pickle(f"backups/eeg_features_alzheimers.pkl")
cntrl_df_test_laptop = pd.read_pickle(f"backups/eeg_features_controls.pkl")

In [7]:
alz_df_test_laptop.head()

,SubjectID,EpochID,WaveBand,Electrode,Power
0,sub-020,ep-0,Delta,Fp1,0.078234
1,sub-020,ep-0,Theta,Fp1,0.006522
2,sub-020,ep-0,Alpha,Fp1,0.003593
3,sub-020,ep-0,Beta,Fp1,0.000334
4,sub-020,ep-0,Total,Fp1,0.011236


In [8]:
# 1. Compare the shapes (rows and columns)
print("Shape comparison:")
print(f"alz_df_test_m1 shape: {alz_df_test_m1.shape}")
print(f"alz_df_test_laptop shape: {alz_df_test_laptop.shape}")

# 2. Check if they have the same columns
print("\nColumn comparison:")
print(f"Same columns: {list(alz_df_test_m1.columns) == list(alz_df_test_laptop.columns)}")
print(f"Columns in m1 but not laptop: {set(alz_df_test_m1.columns) - set(alz_df_test_laptop.columns)}")
print(f"Columns in laptop but not m1: {set(alz_df_test_laptop.columns) - set(alz_df_test_m1.columns)}")

# 3. Compare data values
print("\nValue comparison:")
if alz_df_test_m1.equals(alz_df_test_laptop):
    print("The dataframes have identical values")
else:
    print("The dataframes have different values")


Shape comparison:
alz_df_test_m1 shape: (1856870, 5)
alz_df_test_laptop shape: (1856870, 5)

Column comparison:
Same columns: True
Columns in m1 but not laptop: set()
Columns in laptop but not m1: set()

Value comparison:
The dataframes have different values


In [9]:
# 4. Find differences in values
if not alz_df_test_m1.equals(alz_df_test_laptop) and alz_df_test_m1.shape == alz_df_test_laptop.shape:
    # Check if indices are the same
    print("\nIndex comparison:")
    print(f"Same indices: {alz_df_test_m1.index.equals(alz_df_test_laptop.index)}")
    
    # Find rows that differ
    print("\nRows with differences:")
    diff_mask = ~(alz_df_test_m1 == alz_df_test_laptop).all(axis=1)
    print(f"Number of different rows: {diff_mask.sum()}")
    if diff_mask.sum() > 0:
        print("Sample of different rows:")
        print(alz_df_test_m1[diff_mask].head())



Index comparison:
Same indices: True

Rows with differences:
Number of different rows: 1856870
Sample of different rows:
  SubjectID EpochID WaveBand Electrode     Power
0   sub-020    ep-0    Delta       Fp1  0.078234
1   sub-020    ep-0    Theta       Fp1  0.006522
2   sub-020    ep-0    Alpha       Fp1  0.003593
3   sub-020    ep-0     Beta       Fp1  0.000334
4   sub-020    ep-0    Total       Fp1  0.011236


In [11]:
diff_mask

0          True
1          True
2          True
3          True
4          True
           ... 
1856865    True
1856866    True
1856867    True
1856868    True
1856869    True
Length: 1856870, dtype: bool

In [10]:
# 7. Check for metadata or attribute differences
print("\nDataFrame attribute comparison:")
m1_attrs = set(dir(alz_df_test_m1))
laptop_attrs = set(dir(alz_df_test_laptop))
print(f"Attributes in m1 but not laptop: {m1_attrs - laptop_attrs}")
print(f"Attributes in laptop but not m1: {laptop_attrs - m1_attrs}")

# 8. Check unique subjects (if the subject pools differ)
print("\nSubject comparison:")
print(f"Subjects in m1: {alz_df_test_m1['SubjectID'].nunique()}")
print(f"Subjects in laptop: {alz_df_test_laptop['SubjectID'].nunique()}")
print(f"Same subjects: {set(alz_df_test_m1['SubjectID'].unique()) == set(alz_df_test_laptop['SubjectID'].unique())}")



DataFrame attribute comparison:
Attributes in m1 but not laptop: set()
Attributes in laptop but not m1: set()

Subject comparison:
Subjects in m1: 36
Subjects in laptop: 36
Same subjects: True


In [12]:
# 1. Count total differences and percentage
diff_mask = ~(alz_df_test_m1 == alz_df_test_laptop).all(axis=1)
num_diff_rows = diff_mask.sum()
total_rows = len(alz_df_test_m1)
percent_diff = (num_diff_rows / total_rows) * 100

print(f"Number of different rows: {num_diff_rows}")
print(f"Percentage of rows with differences: {percent_diff:.2f}%")

# 2. Subject analysis
subjects_m1 = alz_df_test_m1['SubjectID'].unique()
subjects_laptop = alz_df_test_laptop['SubjectID'].unique()

print(f"\nNumber of subjects in test_m1: {len(subjects_m1)}")
print(f"Number of subjects in laptop: {len(subjects_laptop)}")

# 3. Check if subjects are the same
subjects_in_both = set(subjects_m1) & set(subjects_laptop)
subjects_only_m1 = set(subjects_m1) - set(subjects_laptop)
subjects_only_laptop = set(subjects_laptop) - set(subjects_m1)

print(f"\nSubjects in both datasets: {len(subjects_in_both)}")
print(f"Subjects only in m1: {subjects_only_m1 if subjects_only_m1 else 'None'}")
print(f"Subjects only in laptop: {subjects_only_laptop if subjects_only_laptop else 'None'}")

# 4. For rows that differ, calculate percentage difference in Power values
if num_diff_rows > 0:
    diff_indices = diff_mask[diff_mask].index
    power_m1 = alz_df_test_m1.loc[diff_indices, 'Power']
    power_laptop = alz_df_test_laptop.loc[diff_indices, 'Power']
    
    # Calculate absolute percentage differences
    abs_percent_diff = abs((power_m1 - power_laptop) / power_laptop * 100)
    
    print(f"\nPower value differences:")
    print(f"Mean absolute % difference: {abs_percent_diff.mean():.2f}%")
    print(f"Max absolute % difference: {abs_percent_diff.max():.2f}%")
    print(f"Min absolute % difference: {abs_percent_diff.min():.2f}%")
    
    # Show examples of rows with largest differences
    largest_diffs = abs_percent_diff.nlargest(3).index
    print("\nExamples of rows with largest differences:")
    for idx in largest_diffs:
        print(f"Row {idx}:")
        print(f"  M1: {alz_df_test_m1.loc[idx].to_dict()}")
        print(f"  Laptop: {alz_df_test_laptop.loc[idx].to_dict()}")


Number of different rows: 1856870
Percentage of rows with differences: 100.00%

Number of subjects in test_m1: 36
Number of subjects in laptop: 36

Subjects in both datasets: 36
Subjects only in m1: None
Subjects only in laptop: None

Power value differences:
Mean absolute % difference: 0.00%
Max absolute % difference: 0.00%
Min absolute % difference: 0.00%

Examples of rows with largest differences:
Row 702158:
  M1: {'SubjectID': 'sub-036', 'EpochID': 'ep-1', 'WaveBand': 'Beta', 'Electrode': 'F3', 'Power': 0.0009767212322952348}
  Laptop: {'SubjectID': 'sub-036', 'EpochID': 'ep-1', 'WaveBand': 'Beta', 'Electrode': 'F3', 'Power': 0.000976721290498972}
Row 1833997:
  M1: {'SubjectID': 'sub-028', 'EpochID': 'ep-308', 'WaveBand': 'Alpha', 'Electrode': 'C3', 'Power': 0.0009775658254458609}
  Laptop: {'SubjectID': 'sub-028', 'EpochID': 'ep-308', 'WaveBand': 'Alpha', 'Electrode': 'C3', 'Power': 0.0009775657672435045}
Row 194052:
  M1: {'SubjectID': 'sub-002', 'EpochID': 'ep-397', 'WaveBand'

In [13]:
# Filter both dataframes to only include subject 'sub-020'
sub_m1 = alz_df_test_m1[alz_df_test_m1['SubjectID'] == 'sub-001']
sub_laptop = alz_df_test_laptop[alz_df_test_laptop['SubjectID'] == 'sub-001']

# Check if they have the same number of rows
print(f"Number of rows for sub-020 in test_m1: {len(sub_m1)}")
print(f"Number of rows for sub-020 in laptop: {len(sub_laptop)}")

# Check if the rows are identical
are_identical = sub_m1.equals(sub_laptop)
print(f"\nAre the data for sub-020 identical? {are_identical}")

# If not identical, find where they differ
if not are_identical:
    # Count rows with differences
    diff_mask = ~(sub_m1 == sub_laptop).all(axis=1)
    num_diff_rows = diff_mask.sum()
    percent_diff = (num_diff_rows / len(sub_m1)) * 100
    
    print(f"\nNumber of different rows for sub-020: {num_diff_rows}")
    print(f"Percentage of rows with differences: {percent_diff:.2f}%")
    
    # Check differences by wave bands
    print("\nDifferences by wave bands:")
    for band in sub_m1['WaveBand'].unique():
        m1_band = sub_m1[sub_m1['WaveBand'] == band]
        laptop_band = sub_laptop[sub_laptop['WaveBand'] == band]
        
        band_diff = ~(m1_band == laptop_band).all(axis=1)
        print(f"  {band}: {band_diff.sum()} rows differ")
        
        if band_diff.sum() > 0:
            # Show a few examples of differences
            print("  Example differences:")
            diff_indices = band_diff[band_diff].index
            for idx in list(diff_indices)[:3]:  # Show up to 3 examples
                row_m1 = m1_band.loc[idx]
                row_laptop = laptop_band.loc[idx]
                print(f"    Row {idx}:")
                print(f"      M1: EpochID={row_m1['EpochID']}, Power={row_m1['Power']}")
                print(f"      Laptop: EpochID={row_laptop['EpochID']}, Power={row_laptop['Power']}")
                print(f"      Difference: {abs(row_m1['Power'] - row_laptop['Power'])}")


Number of rows for sub-020 in test_m1: 37810
Number of rows for sub-020 in laptop: 37810

Are the data for sub-020 identical? False

Number of different rows for sub-020: 37810
Percentage of rows with differences: 100.00%

Differences by wave bands:
  Delta: 7562 rows differ
  Example differences:
    Row 573420:
      M1: EpochID=ep-0, Power=0.08257892642542033
      Laptop: EpochID=ep-0, Power=0.08257892727851868
      Difference: 8.530983419685612e-10
    Row 573425:
      M1: EpochID=ep-0, Power=0.08389220643958975
      Laptop: EpochID=ep-0, Power=0.08389220386743546
      Difference: 2.572154295110707e-09
    Row 573430:
      M1: EpochID=ep-0, Power=0.08212274846138015
      Laptop: EpochID=ep-0, Power=0.08212275058031082
      Difference: 2.1189306698143895e-09
  Theta: 7562 rows differ
  Example differences:
    Row 573421:
      M1: EpochID=ep-0, Power=0.005611645685714301
      Laptop: EpochID=ep-0, Power=0.005611645523458719
      Difference: 1.6225558136656604e-10
    Row 